# BNOT quickstart

This notebook exercises the shipped `ibnot_new_cli` backend through the Python wrapper.

In [ ]:
from pathlib import Path
import importlib.util
import inspect
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "ibnot_cli").exists() and (path / "python").exists()
)
API_PATH = REPO_ROOT / "python" / "src" / "ibnot_cli_wrapper" / "api.py"
spec = importlib.util.spec_from_file_location("ibnot_cli_wrapper_repo_api", API_PATH)
ibnot_api = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = ibnot_api
assert spec.loader is not None
spec.loader.exec_module(ibnot_api)

find_default_executable = ibnot_api.find_default_executable
make_linear_ramp = ibnot_api.make_linear_ramp
make_sine_landscape = ibnot_api.make_sine_landscape
make_uniform = ibnot_api.make_uniform
NativeConfig = ibnot_api.NativeConfig
RenderConfig = ibnot_api.RenderConfig
OutputConfig = ibnot_api.OutputConfig
InferenceRequest = ibnot_api.InferenceRequest
run_inference = ibnot_api.run_inference

OUTPUT_DIR = REPO_ROOT / "python" / "notebooks" / "_generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SIZE = 512
NUM_SITES = 1024
SEED = 7
OUTPUT_WIDTH = SIZE
OUTPUT_HEIGHT = SIZE
POINT_RADIUS = 0.002
DPI = 300
DENSITY_INVERT = True
NATIVE_TIMER = True
RENDER_ENABLED = True
PNG_ENABLED = True
KEEP_PGM = True
KEEP_EPS = True
KEEP_STATS = True

CLI = find_default_executable()
GS = shutil.which("gs")
if GS is None and PNG_ENABLED:
    raise RuntimeError("Ghostscript ('gs') is required for notebook PNG previews.")

def build_request(image: np.ndarray, stem: str, *, invert: bool = False):
    render = RenderConfig(
        enabled=RENDER_ENABLED,
        render_width=OUTPUT_WIDTH if RENDER_ENABLED else None,
        render_height=OUTPUT_HEIGHT if RENDER_ENABLED else None,
        point_radius=POINT_RADIUS,
        png_enabled=PNG_ENABLED,
        dpi=DPI,
    )
    native = NativeConfig(
        num_sites=NUM_SITES,
        seed=SEED,
        max_iters=25,
        max_newton_iters=50,
        invert=invert,
        native_timer=NATIVE_TIMER,
    )
    output = OutputConfig(
        output_dir=OUTPUT_DIR,
        output_stem=stem,
        keep_pgm=KEEP_PGM,
        keep_eps=KEEP_EPS,
        keep_stats_txt=KEEP_STATS,
    )
    return InferenceRequest(image_array=image, native=native, render=render, output=output, executable=CLI)

print(f"Repo root: {REPO_ROOT}")
print(f"API path: {API_PATH}")
print(f"run_inference signature: {inspect.signature(run_inference)}")
print(f"CLI: {CLI}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Uniform invert: {UNIFORM_INVERT}")
print(f"Non-uniform invert: {DENSITY_INVERT}")

def show_case(title: str, image: np.ndarray, result):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image, cmap="gray", vmin=0.0, vmax=1.0)
    axes[0].set_title(f"{title} input")
    axes[0].axis("off")

    if result.png_path is not None:
        rendered = plt.imread(result.png_path)
        axes[1].imshow(rendered)
        axes[1].set_title(f"{title} output")
        axes[1].axis("off")
    else:
        axes[1].text(0.5, 0.5, "No PNG output", ha="center", va="center")
        axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    print("stats:", result.stats)
    print("timings:", result.timings)
    if result.warnings:
        print("warnings:", result.warnings)


In [ ]:
uniform = make_uniform(size=SIZE, value=1.0)
uniform_result = run_inference(build_request(uniform, "uniform_512_1024", invert=False))
show_case("Uniform", uniform, uniform_result)


In [ ]:
ramp = make_linear_ramp(size=SIZE, left=1.0, right=0.0)
ramp_result = run_inference(build_request(ramp, "ramp_512_1024", invert=DENSITY_INVERT))
show_case("Ramp", ramp, ramp_result)


In [ ]:
sine = make_sine_landscape(size=SIZE, fx=4, fy=4)
sine_result = run_inference(build_request(sine, "sine2d_512_1024", invert=DENSITY_INVERT))
show_case("2D sine", sine, sine_result)
